In [ ]:
# CELL 24 — GNN setup on the full dataset: load graphs, consistency node features, normalization, splits (train / val-from-train / dev), model + training helpers
!pip install -q torch_geometric

import os, glob, gzip, pickle, time, zlib, random, warnings, gc
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, GINEConv
from torch_geometric.utils import scatter
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
torch.set_num_threads(4)

hg = sorted(glob.glob("/kaggle/input/**/full/full_graphs.pkl.gz", recursive=True))
hf = sorted(glob.glob("/kaggle/input/**/full/full_features.csv", recursive=True))
assert hg and hf, "full_graphs.pkl.gz / full_features.csv not found - is BTP-full attached as input?"
GRAPHS, FEATS = hg[0], hf[0]
RES_DIR = "/kaggle/working/gnn_full"
os.makedirs(RES_DIR, exist_ok=True)
print("Graphs:", GRAPHS)

t0 = time.time()
with gzip.open(GRAPHS, "rb") as fh:
    G = pickle.load(fh)
graphs, FEAT, NE = G["graphs"], list(G["feat_names"]), len(G["edge_keys"])
del G
feat_df = pd.read_csv(FEATS).set_index("out_file")
print(f"Loaded {len(graphs)} graphs in {time.time() - t0:.0f}s | base node features: {len(FEAT)} | edge types: {NE}")

# ---------- splits: train (minus 15% for validation) / val (from train docs) / dev (reporting only) ----------
info = pd.DataFrame([{k: g[k] for k in ["out_file", "orig_file", "family", "variant", "label", "tool", "twin",
                                         "edit_type", "edit_page", "split"]} for g in graphs])
info["is_val"] = (info.split == "train") & info.orig_file.map(lambda s: zlib.crc32((s + "|val").encode()) % 100 < 15)
tr_idx = info.index[(info.split == "train") & ~info.is_val].tolist()
va_idx = info.index[info.is_val].tolist()
te_idx = info.index[info.split == "dev"].tolist()
print(f"Graphs -> train: {len(tr_idx)} | val: {len(va_idx)} | dev: {len(te_idx)}")
print(f"Docs   -> train: {info.loc[tr_idx].orig_file.nunique()} | val: {info.loc[va_idx].orig_file.nunique()}"
      f" | dev: {info.loc[te_idx].orig_file.nunique()}")

# ---------- consistency node features (same as pilot CELL 14) ----------
C_CONTENT = FEAT.index("t_content")
FP = ["f_flate", "f_lzw", "f_dct", "f_none", "f_other", "h_789C", "h_78DA", "h_7801", "h_785E", "h_other",
      "zlevel", "zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio", "len_indirect", "entropy", "raw_len_log"]
SIG = ["f_flate", "f_lzw", "f_dct", "f_none", "f_other", "h_789C", "h_78DA", "h_7801", "h_785E", "h_other",
       "zlevel", "zlib_repro", "adler_ok", "sync_end", "tail_junk", "len_indirect"]
FP_I = [FEAT.index(c) for c in FP]
SIG_I = [FEAT.index(c) for c in SIG]
EXTRA = [f"dev_{c}" for c in FP] + ["odd_one_out", "is_singleton", "n_content_log"]

def consistency(X):
    n = X.shape[0]
    E = np.zeros((n, len(EXTRA)), dtype=np.float32)
    idx = np.where(X[:, C_CONTENT] > 0.5)[0]
    k = len(idx)
    E[:, -1] = np.log1p(k) / 5
    if k >= 2:
        S = X[idx][:, FP_I]
        tot = S.sum(0)
        E[idx, :len(FP)] = np.abs(S - (tot - S) / (k - 1))
        sig = np.round(X[idx][:, SIG_I], 3)
        _, inv, cnt = np.unique(sig, axis=0, return_inverse=True, return_counts=True)
        inv = np.asarray(inv).reshape(-1)
        E[idx, len(FP)] = 1 - cnt[inv] / k
        if k >= 3:
            E[idx, len(FP) + 1] = (cnt[inv] == 1).astype(np.float32)
    return E

ALL_NAMES = FEAT + EXTRA
BASE_COLS = list(range(len(FEAT)))
ALL_COLS = list(range(len(ALL_NAMES)))
tr_set = set(tr_idx)
XS = [None] * len(graphs)
S1 = np.zeros(len(ALL_NAMES)); S2 = np.zeros(len(ALL_NAMES)); NN = 0
for i, g in enumerate(tqdm(graphs, desc="Consistency features")):
    X = g["x"].astype(np.float32)
    if X.shape[0] == 0:
        X = np.zeros((1, len(FEAT)), dtype=np.float32)
    XA = np.concatenate([X, consistency(X)], 1)
    XS[i] = XA
    g["x"] = None
    if i in tr_set:
        S1 += XA.sum(0, dtype=np.float64)
        S2 += (XA.astype(np.float64) ** 2).sum(0)
        NN += XA.shape[0]
MU = (S1 / NN).astype(np.float32)
SD = np.sqrt(np.maximum(S2 / NN - (S1 / NN) ** 2, 0)).astype(np.float32)
SD[SD < 1e-6] = 1.0
print("Training nodes used for normalization:", NN)

# ---------- document-level features for CGNN_DOC (standardized on train graphs) ----------
META = ["has_producer", "producer_len", "moddate_eq_creation", "has_xmp", "tool_word_in_file"]
L1 = ["size_log", "version", "binary_comment", "n_eof", "n_startxref", "xref_table", "xref_stream", "obj_streams",
      "linearized", "true_revisions", "eol_crlf", "eol_cr", "eol_lf", "n_obj_defs", "obj_order_increasing",
      "obj_gen_nonzero", "obj_max_over_count", "obj_redefined", "obj_newline_dict", "dict_open_space",
      "dict_close_space", "stream_kw_crlf", "endobj_newline_before", "reals_dot_zero", "has_ID", "tool_word_in_file", "n_pages"]
L23 = ["cs_n", "cs_frac_flate", "cs_frac_lzw", "cs_frac_none", "cs_n_filters", "cs_frac_zlib_repro", "cs_n_hdr",
       "cs_n_zlevel", "cs_frac_sync", "cs_frac_adler", "cs_frac_junk", "cs_frac_len_indirect", "cs_ratio_std",
       "cs_ratio_max_z", "cs_n_sig", "cs_majority_share", "cs_n_singletons"]
L4 = ["upd_n_objs", "upd_touch_content", "upd_has_annot", "upd_has_page", "upd_has_sig"]
L5 = ["l5_n_annots", "l5_n_freetext", "l5_n_text_notes", "l5_n_img_xobj", "l5_n_fonts", "l5_n_std14_unembedded",
      "l5_frac_pages_multi_stream", "l5_max_streams_per_page"]
FULL = sorted(set(META + L1 + L23 + L4 + L5))
DOC = feat_df.loc[info.out_file.values, FULL].astype(float).values
dmu, dsd = np.nanmean(DOC[tr_idx], 0), np.nanstd(DOC[tr_idx], 0)
dsd[dsd < 1e-6] = 1.0
DOC = np.nan_to_num((DOC - dmu) / dsd, nan=0.0).astype(np.float32)

# ---------- PyG datasets (built once; raw arrays freed as we go) ----------
DATA = []
for i, g in enumerate(tqdm(graphs, desc="Building graphs")):
    XA = XS[i]
    ei = g["edge_index"].astype(np.int64)
    et = g["edge_type"].astype(np.int64)
    if ei.size:
        ei_t = torch.tensor(np.concatenate([ei, ei[::-1]], 1))
        et_t = torch.tensor(np.concatenate([et, et + NE]))
    else:
        ei_t, et_t = torch.zeros((2, 0), dtype=torch.long), torch.zeros(0, dtype=torch.long)
    npg = g["node_page"].astype(np.int64) if len(g["node_page"]) == XA.shape[0] else np.full(XA.shape[0], -1)
    DATA.append(Data(x=torch.tensor((XA - MU) / SD, dtype=torch.float32), edge_index=ei_t, edge_type=et_t,
                     y=torch.tensor([float(g["label"])]), node_page=torch.tensor(npg),
                     is_content=torch.tensor(XA[:, C_CONTENT] > 0.5),
                     doc=torch.tensor(DOC[i]).unsqueeze(0)))
    XS[i] = None
    g["edge_index"] = g["edge_type"] = None
del XS
gc.collect()
print("Node features after consistency:", len(ALL_COLS), "| document features:", DOC.shape[1])

# ---------- the model (identical to pilot CELL 14) ----------
R = 5.0
class Net(nn.Module):
    def __init__(self, cols, relational, content_pool, node_head, use_doc, hid=64, layers=3, drop=0.2):
        super().__init__()
        self.register_buffer("cols", torch.tensor(cols, dtype=torch.long))
        self.relational, self.content_pool, self.node_head, self.use_doc, self.drop = relational, content_pool, node_head, use_doc, drop
        self.inp = nn.Sequential(nn.Linear(len(cols), hid), nn.ReLU())
        if relational:
            self.etype = nn.Embedding(2 * NE, 16)
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        for _ in range(layers):
            mlp = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, hid))
            self.convs.append(GINEConv(mlp, edge_dim=16) if relational else GINConv(mlp))
            self.norms.append(nn.LayerNorm(hid))
        d_cat = hid * (layers + 1)
        d_pool = d_cat * (4 if content_pool else 2) + (hid if use_doc else 0)
        if use_doc:
            self.doc_mlp = nn.Sequential(nn.Linear(DOC.shape[1], hid), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(d_pool, hid), nn.ReLU(), nn.Dropout(drop), nn.Linear(hid, 1))
        if node_head:
            self.node_mlp = nn.Sequential(nn.Linear(d_cat, hid), nn.ReLU(), nn.Linear(hid, 1))

    def forward(self, b):
        h = self.inp(b.x[:, self.cols])
        hs = [h]
        ea = self.etype(b.edge_type) if self.relational else None
        for conv, norm in zip(self.convs, self.norms):
            m = conv(h, b.edge_index, ea) if self.relational else conv(h, b.edge_index)
            h = h + F.dropout(F.relu(norm(m)), self.drop, self.training)
            hs.append(h)
        H = torch.cat(hs, 1)
        B = b.num_graphs
        parts = [scatter(H, b.batch, 0, dim_size=B, reduce="mean"), scatter(H, b.batch, 0, dim_size=B, reduce="max")]
        cm = b.is_content
        if self.content_pool:
            parts += [scatter(H[cm], b.batch[cm], 0, dim_size=B, reduce="mean"),
                      scatter(H[cm], b.batch[cm], 0, dim_size=B, reduce="max")]
        gvec = torch.nan_to_num(torch.cat(parts, 1), nan=0.0, posinf=0.0, neginf=0.0)
        if self.use_doc:
            gvec = torch.cat([gvec, self.doc_mlp(b.doc)], 1)
        logit = self.head(gvec).squeeze(-1)
        z = None
        if self.node_head:
            z = self.node_mlp(H).squeeze(-1)
            zc, bc = z[cm], b.batch[cm]
            cnt = scatter(torch.ones_like(zc), bc, 0, dim_size=B, reduce="sum")
            mx = torch.nan_to_num(scatter(zc, bc, 0, dim_size=B, reduce="max"), nan=0.0, posinf=0.0, neginf=0.0)
            s = scatter(torch.exp(R * (zc - mx[bc])), bc, 0, dim_size=B, reduce="mean")
            lse = mx + torch.log(s + 1e-12) / R
            logit = logit + torch.where(cnt > 0, lse, torch.zeros_like(lse))
        return logit, z

def make_model(kind):
    if kind == "GIN":
        return Net(BASE_COLS, relational=False, content_pool=False, node_head=False, use_doc=False)
    if kind == "CGNN":
        return Net(ALL_COLS, relational=True, content_pool=True, node_head=True, use_doc=False)
    if kind == "CGNN_DOC":
        return Net(ALL_COLS, relational=True, content_pool=True, node_head=True, use_doc=True)
    raise ValueError(kind)

@torch.no_grad()
def predict(model, items, want_pages=False):
    model.eval()
    S, P = [], []
    for b in DataLoader(items, batch_size=64, shuffle=False):
        out, z = model(b)
        S += torch.sigmoid(out).tolist()
        if want_pages:
            ptr = b.ptr.tolist()
            for a, c in zip(ptr[:-1], ptr[1:]):
                if z is None:
                    P.append(-1)
                    continue
                npg, zz = b.node_page[a:c], z[a:c]
                m = npg >= 0
                P.append(int(npg[m][torch.argmax(zz[m])]) if bool(m.any()) else -1)
    return np.array(S), P

def train_run(kind, seed, max_epochs=60, patience=10, lr=2e-3, bs=32):
    path = f"{RES_DIR}/{kind}_seed{seed}.csv"
    if os.path.exists(path):
        print(f"{kind} seed {seed}: already done, skipping ({path})")
        return
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = make_model(kind)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
    loader = DataLoader([DATA[i] for i in tr_idx], batch_size=bs, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    va_items = [DATA[i] for i in va_idx]
    yva = info.label.values[va_idx]
    best, best_state, best_ep, bad, log = -1.0, None, 0, 0, []
    t0 = time.time()
    for ep in range(1, max_epochs + 1):
        model.train()
        tot = 0.0
        for b in tqdm(loader, desc=f"{kind} seed {seed} | epoch {ep}", leave=False, mininterval=30):
            opt.zero_grad()
            out, _ = model(b)
            loss = F.binary_cross_entropy_with_logits(out, b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tot += loss.item() * b.num_graphs
        va, _ = predict(model, va_items)
        auc = roc_auc_score(yva, va)
        sched.step(auc)
        if auc > best + 1e-4:
            best, best_state, best_ep, bad = auc, deepcopy(model.state_dict()), ep, 0
        else:
            bad += 1
        log.append({"epoch": ep, "loss": tot / len(tr_idx), "val_auc": auc, "lr": opt.param_groups[0]["lr"]})
        print(f"{kind} seed {seed} | epoch {ep:2d} | loss {tot / len(tr_idx):.4f} | val AUC {auc:.4f} | "
              f"best {best:.4f}@{best_ep} | {(time.time() - t0) / 60:.1f} min", flush=True)
        if bad >= patience:
            break
    model.load_state_dict(best_state)
    sc, pages = predict(model, [DATA[i] for i in te_idx], want_pages=True)
    out = info.loc[te_idx, ["out_file"]].copy()
    out["score"], out["pred_page"] = sc, pages
    out["kind"], out["seed"], out["best_val_auc"], out["best_epoch"] = kind, seed, best, best_ep
    pd.DataFrame(log).to_csv(f"{RES_DIR}/{kind}_seed{seed}_log.csv", index=False)
    out.to_csv(path, index=False)
    print(f"{kind} seed {seed}: best val AUC {best:.4f} at epoch {best_ep} | {(time.time() - t0) / 60:.1f} min -> saved {path}")

print(f"\nSetup complete in {(time.time() - t0) / 60:.1f} min. Next: CELL 25 (training).")

In [ ]:
# CELL 25 — Train ONE GNN kind for 3 seeds. Set KIND per notebook: "GIN" (BTP-gnn-GIN), "CGNN" (BTP-gnn-CGNN), "CGNN_DOC" (BTP-gnn-DOC)
KIND = "CGNN"

for s in [0, 1, 2]:
    train_run(KIND, s)
print("\nFinished files:", sorted(os.listdir(RES_DIR)))